# Cats vs Dogs Classification using CNN

This notebook builds a Convolutional Neural Network (CNN) to classify images into two classes: **Cat** and **Dog**.

**Environment:** Kaggle Notebook

**Dataset:** Dog and Cat Classification Dataset from Kaggle

### Workflow
1. Import the required libraries
2. Load and preprocess the images
3. Apply normalization and data augmentation
4. Load training and validation data
5. Build the CNN architecture
6. Compile and train the model
7. Evaluate the model
8. Visualize accuracy and loss
9. Test the model on a single image

# Cats vs Dogs Classification using CNN
Dataset: https://www.kaggle.com/datasets/bhavikjikadara/dog-and-cat-classification-dataset

Steps:
1. Import libraries
2. Load images and apply preprocessing (normalization + augmentation)
3. Build a CNN model
4. Train the model
5. Evaluate the results

## 1. Import Libraries

The notebook imports NumPy and Matplotlib together with TensorFlow/Keras tools for image preprocessing, CNN construction, training, and evaluation.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

## 2. Dataset Path

The dataset contains two class folders: `Cat` and `Dog`. The dataset path should point to the directory containing these folders.

> **Kaggle note:** When running this notebook on Kaggle, the dataset path should match the location of the attached Kaggle dataset.

## 1. Dataset path
After downloading the dataset from Kaggle, you will have a main folder containing two sub-folders: `Cat` and `Dog`.
Put that main folder's path below (same idea as the `pd.read_csv(...)` path in the previous assignment).

In [ ]:
dataset_dir = 'D:\\Instant AI\\AI\\Assignments\\Assignment 18\\PetImages'  # <-- change this to your dataset path

img_size = 128       # all images will be resized to 128x128
batch_size = 32

## 3. Pre-processing: Normalization + Data Augmentation

- `rescale=1./255` normalizes pixel values from the original 0–255 range to 0–1.
- Rotation, shifting, shearing, zooming, and horizontal flipping create variations of training images.
- `validation_split=0.2` reserves 20% of the available images for validation.

## 2. Pre-processing: Normalization + Data Augmentation
- `rescale=1./255` -> normalization (pixel values become between 0 and 1 instead of 0-255)
- `rotation_range`, `zoom_range`, `horizontal_flip`, etc. -> data augmentation (creates modified copies of images so the model generalizes better)
- `validation_split=0.2` -> 80% of the images for training, 20% for validation (same purpose as `train_test_split` before)

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,          # normalization
    rotation_range=20,       # augmentation: rotate images randomly
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2
)

# validation data should only be normalized, not augmented
val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

## 4. Load Images from the Folders

`flow_from_directory` loads images directly from the class folders and generates binary labels for the two classes.

## 3. Load images from the folders
`flow_from_directory` reads the images directly from the `Cat` and `Dog` folders and labels them automatically (Cat=0, Dog=1).

In [ ]:
train_generator = train_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='training'
)

val_generator = val_datagen.flow_from_directory(
    dataset_dir,
    target_size=(img_size, img_size),
    batch_size=batch_size,
    class_mode='binary',
    subset='validation'
)

## 5. Build the CNN Model

The CNN uses convolutional layers to learn visual features and max-pooling layers to reduce spatial dimensions. Dense layers then use the learned features for classification.

Because this is a binary classification problem, the final layer uses one neuron with `sigmoid` activation.

## 4. Build the CNN model
A simple CNN: a few Conv2D + MaxPooling layers to extract features, then Dense layers to classify.
The last layer has 1 neuron with `sigmoid` activation because this is binary classification (Cat vs Dog).

In [ ]:
cnn_model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(img_size, img_size, 3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),           # dropout to reduce overfitting
    Dense(1, activation='sigmoid')   # binary output: 0 = Cat, 1 = Dog
])

cnn_model.summary()

## 6. Compile the Model

The model is compiled with the Adam optimizer, binary cross-entropy loss, and accuracy as the training metric.

## 5. Compile the model

In [ ]:
cnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

## 7. Train the Model

The CNN is trained using the training generator and evaluated on the validation generator during training.

`EarlyStopping` monitors validation loss and restores the best weights when validation performance stops improving.

## 6. Train the model
`EarlyStopping` stops training automatically if the validation loss stops improving, to save time and avoid overfitting.

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

history = cnn_model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=15,
    callbacks=[early_stop]
)

## 8. Evaluate the Model

The trained model is evaluated on the validation data to obtain validation loss and validation accuracy.

## 7. Evaluate the model

In [ ]:
val_loss, val_accuracy = cnn_model.evaluate(val_generator)
print(f"Validation Accuracy: {val_accuracy:.4f}")
print(f"Validation Loss: {val_loss:.4f}")

## 9. Plot Accuracy and Loss Curves

Training and validation curves help visualize how the model behaves across epochs and can reveal signs of overfitting or underfitting.

## 8. Plot accuracy and loss curves

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Accuracy over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Loss over Epochs')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()

## 10. Predict on a Single Image

This optional section loads one image, applies the same image size used during training, and uses the trained CNN to predict whether the image belongs to the Cat or Dog class.

## 9. Predict on a single image (optional test)

In [ ]:
from tensorflow.keras.preprocessing import image

img_path = 'path_to_a_test_image.jpg'   # <-- put any test image path here

img = image.load_img(img_path, target_size=(img_size, img_size))
img_array = image.img_to_array(img) / 255.0     # same normalization as training
img_array = np.expand_dims(img_array, axis=0)

prediction = cnn_model.predict(img_array)[0][0]

label = 'Dog' if prediction > 0.5 else 'Cat'
print(f"Prediction: {label} (score={prediction:.4f})")